# LeetCode 3269 — Constructing Two Increasing Arrays bằng Python

- Link: https://leetcode.com/problems/constructing-two-increasing-arrays/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **DP trộn hai dãy** (`f[i][j]` = số lớn nhất nhỏ nhất sau khi điền `i` phần tử đầu của `nums1` và `j` phần tử đầu của `nums2`), hàm "số **kế tiếp** cùng tính chẵn lẻ"

> **Ghi chú Premium:** bài Premium nên LeetCode không trả đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `minLargest(self, nums1, nums2)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3269/) — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def minLargest(self, nums1: List[int], nums2: List[int]) -> int:
```

## 1. Mục tiêu học

1. Thay mỗi `0` bằng số **chẵn dương**, mỗi `1` bằng số **lẻ dương**; hai mảng đều **tăng nghiêm ngặt**; mọi số dùng tối đa một lần (giữa cả hai mảng). Cực tiểu hóa số lớn nhất.
2. Quan sát: sắp **tất cả** các số đã dùng tăng dần ⇒ được một cách **trộn** hai mảng. Nếu biết thứ tự trộn, điền tham lam "số nhỏ nhất hợp lệ lớn hơn số vừa dùng" là tối ưu.
3. DP trên thứ tự trộn: `f[i][j] = min(next(f[i−1][j], nums1[i−1]), next(f[i][j−1], nums2[j−1]))` với `next(v, b)` = số nhỏ nhất `> v` có tính chẵn lẻ `b`.
4. `O(m · n)`, có thể lăn một chiều.
5. Kiểm chứng bằng vét cạn gán số `1..M`.

## 2. Đề bài & ví dụ

Cho hai mảng `nums1`, `nums2` chỉ gồm `0` và `1`. Thay mỗi `0` bằng một số nguyên dương **chẵn** và mỗi `1` bằng một số nguyên dương **lẻ**, sao cho cả hai mảng **tăng** và mỗi số nguyên được dùng **tối đa một lần**. Trả về giá trị nhỏ nhất có thể của số lớn nhất trong hai mảng.

**Ví dụ 1:** `nums1 = []`, `nums2 = [1,0,1,1]` → `5` (`[1,2,3,5]`).
**Ví dụ 2:** `nums1 = [0,1,0,1]`, `nums2 = [1,0,0,1]` → `9` (`[2,3,8,9]`, `[1,4,6,7]`).
**Ví dụ 3:** `nums1 = [0,1,0,0,1]`, `nums2 = [0,0,0,1]` → `13`.

**Ràng buộc (theo doocs):** `0 ≤ len(nums1) ≤ 1000`; `1 ≤ len(nums2) ≤ 1000`.

In [1]:
from typing import List
from functools import lru_cache

examples = [([], [1, 0, 1, 1], 5), ([0, 1, 0, 1], [1, 0, 0, 1], 9), ([0, 1, 0, 0, 1], [0, 0, 0, 1], 13)]
for a, b, exp in examples:
    print(a, b, '-> kỳ vọng', exp)

[] [1, 0, 1, 1] -> kỳ vọng 5
[0, 1, 0, 1] [1, 0, 0, 1] -> kỳ vọng 9
[0, 1, 0, 0, 1] [0, 0, 0, 1] -> kỳ vọng 13


## 3. Kiến thức nền

### 3.1. Số kế tiếp cùng tính chẵn lẻ

`next(v, b)` = số nhỏ nhất `> v` có `x mod 2 = b`: nếu `v + 1` đúng tính chẵn lẻ thì `v + 1`, ngược lại `v + 2`.

In [2]:
def nxt(v, b):
    return v + 1 if (v + 1) % 2 == b else v + 2

print([nxt(v, 0) for v in range(5)], [nxt(v, 1) for v in range(5)])

[2, 2, 4, 4, 6] [1, 3, 3, 5, 5]


Lưu ý `next(0, 0) = 2` (số chẵn **dương** nhỏ nhất), `next(0, 1) = 1`.

### 3.2. Trộn hai dãy

Mọi cách gán hợp lệ, khi xếp các số đã dùng tăng dần, cho một dãy trộn (giữ thứ tự trong từng mảng). Ngược lại, với một dãy trộn cố định, gán tham lam theo thứ tự trộn cho kết quả tốt nhất ứng với dãy trộn đó (mỗi bước dùng số nhỏ nhất có thể).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: nếu xét riêng, `nums1 = [0,1,0,1]` cần `[2,3,4,5]`, `nums2 = [1,0,0,1]` cần `[1,2,4,5]` — nhưng các số trùng nhau! Phải trộn: dãy chẵn lẻ cần dùng tổng cộng có 4 số chẵn (`2, 4, 6, 8`) và 4 số lẻ, nên số lớn nhất ít nhất là 8 hoặc 9; thứ tự trong mỗi mảng ép thêm ràng buộc. Đáp án `9`.

## 5. Cách 1 — Brute Force: gán số `1..M` (bỏ qua hoặc dùng cho mảng nào), tăng dần `M`

In [3]:
def brute(a, b):
    m, n = len(a), len(b)
    def feasible(M):
        @lru_cache(maxsize=None)
        def go(v, i, j):                      # đang xét số v, đã điền i phần tử của a, j của b
            if i == m and j == n:
                return True
            if v > M:
                return False
            if go(v + 1, i, j):
                return True
            if i < m and v % 2 == a[i] and go(v + 1, i + 1, j):
                return True
            if j < n and v % 2 == b[j] and go(v + 1, i, j + 1):
                return True
            return False
        return go(1, 0, 0)
    M = 1
    while not feasible(M):
        M += 1
    return M

for a, b, exp in examples:
    print(brute(a, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
9 | kỳ vọng 9
13 | kỳ vọng 13


Oracle này không dùng ý tưởng "trộn + tham lam" — nó thử mọi cách dùng các số `1..M`.

## 6. Ý tưởng thuật toán / bước đột phá

```
f[0][0] = 0
f[i][0] = next(f[i−1][0], a[i−1]);   f[0][j] = next(f[0][j−1], b[j−1])
f[i][j] = min(next(f[i−1][j], a[i−1]), next(f[i][j−1], b[j−1]))
return f[m][n]
```

`f[i][j]` = số lớn nhất (cũng là số dùng sau cùng) nhỏ nhất khi đã điền `i` phần tử đầu của `a` và `j` phần tử đầu của `b`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ

In [4]:
import sys
sys.setrecursionlimit(10000)
def memo_solution(a, b):
    @lru_cache(maxsize=None)
    def f(i, j):
        if i == 0 and j == 0:
            return 0
        best = float('inf')
        if i:
            best = min(best, nxt(f(i - 1, j), a[i - 1]))
        if j:
            best = min(best, nxt(f(i, j - 1), b[j - 1]))
        return best
    ans = f(len(a), len(b))
    f.cache_clear()
    return ans

for a, b, exp in examples:
    print(memo_solution(a, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
9 | kỳ vọng 9
13 | kỳ vọng 13


### 7.2. Cách 3 — DP bảng `(m + 1) × (n + 1)` (lời giải chính)

In [5]:
def table_dp(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = nxt(f[i - 1][0], a[i - 1])
    for j in range(1, n + 1):
        f[0][j] = nxt(f[0][j - 1], b[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = min(nxt(f[i - 1][j], a[i - 1]), nxt(f[i][j - 1], b[j - 1]))
    return f[m][n]

for a, b, exp in examples:
    print(table_dp(a, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
9 | kỳ vọng 9
13 | kỳ vọng 13


### 7.3. Cách 4 — DP lăn một chiều

Hàng `i` chỉ cần hàng `i − 1` ⇒ bộ nhớ `O(n)`.

In [6]:
def rolling_dp(a, b):
    n = len(b)
    row = [0] * (n + 1)
    for j in range(1, n + 1):
        row[j] = nxt(row[j - 1], b[j - 1])
    for x in a:
        new = [nxt(row[0], x)] + [0] * n
        for j in range(1, n + 1):
            u = nxt(row[j], x)
            v = nxt(new[j - 1], b[j - 1])
            new[j] = u if u < v else v
        row = new
    return row[n]

for a, b, exp in examples:
    print(rolling_dp(a, b), '| kỳ vọng', exp)

5 | kỳ vọng 5
9 | kỳ vọng 9
13 | kỳ vọng 13


## 8. TRACE từng bước

In bảng `f` cho ví dụ 2.

In [7]:
def trace(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = nxt(f[i - 1][0], a[i - 1])
    for j in range(1, n + 1):
        f[0][j] = nxt(f[0][j - 1], b[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = min(nxt(f[i - 1][j], a[i - 1]), nxt(f[i][j - 1], b[j - 1]))
    print('       b:', ['-'] + b)
    for i in range(m + 1):
        print(f"a = {'-' if i == 0 else a[i - 1]}:", f[i])
    print('đáp án:', f[m][n], '| vét cạn:', brute(a, b))

trace(*examples[1][:2])

       b: ['-', 1, 0, 0, 1]
a = -: [0, 1, 2, 4, 5]
a = 0: [2, 2, 4, 6, 6]
a = 1: [3, 3, 4, 6, 7]
a = 0: [4, 4, 6, 8, 8]
a = 1: [5, 5, 6, 8, 9]
đáp án: 9 | vét cạn: 9


## 9. Code nộp LeetCode

(Chữ ký theo doocs — xem ghi chú Premium.)

In [8]:
class Solution:
    def minLargest(self, nums1: List[int], nums2: List[int]) -> int:
        def nx(v, p):
            return v + 1 if (v + 1) & 1 == p else v + 2

        n = len(nums2)
        row = [0] * (n + 1)
        for j in range(1, n + 1):
            row[j] = nx(row[j - 1], nums2[j - 1])
        for x in nums1:
            new = [0] * (n + 1)
            new[0] = nx(row[0], x)
            for j in range(1, n + 1):
                u = nx(row[j], x)
                v = nx(new[j - 1], nums2[j - 1])
                new[j] = u if u < v else v
            row = new
        return row[n]

for a, b, exp in examples:
    got = Solution().minLargest(a, b)
    print(got, 'OK' if got == exp else 'SAI')

5 OK
9 OK
13 OK


## 10. Vì sao thuật toán đúng

1. **Từ cách gán sang dãy trộn:** sắp các số đã dùng tăng dần; phần tử `k` của `a` (hoặc `b`) xuất hiện đúng theo thứ tự trong mảng của nó ⇒ một dãy trộn.
2. **Tham lam theo dãy trộn là tối ưu:** với một dãy trộn cố định, quy nạp: sau `t` bước, số dùng ở bước `t` của cách tham lam `≤` số dùng ở bước `t` của mọi cách gán theo cùng dãy trộn (vì `next` đơn điệu không giảm theo `v`).
3. **DP trên dãy trộn:** trạng thái `(i, j)` — đã điền `i` phần tử của `a` và `j` của `b` — với giá trị là số dùng gần nhất nhỏ nhất. Bước cuối thuộc `a` hoặc `b`; nhờ tính đơn điệu của `next`, giữ giá trị **nhỏ nhất** của mỗi trạng thái là đủ (trạng thái nhỏ hơn không bao giờ tệ hơn).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn số `1..M` | `O(M² · m · n)` | `O(M · m · n)` |
| 2. Đệ quy có nhớ | `O(m n)` | `O(m n)` |
| 3. Bảng DP | `O(m n)` | `O(m n)` |
| 4. DP lăn | `O(m n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo': memo_solution, 'table': table_dp, 'rolling': rolling_dp, 'submit': lambda a, b: Solution().minLargest(a, b)}
edge = [([], [0], 2), ([], [1], 1), ([1], [1], 3), ([0], [0], 4), ([0, 0], [1, 1], 4), ([], [0, 0, 0], 6)]
for a, b, exp in examples + edge:
    assert brute(a, b) == exp, (a, b, brute(a, b))
    for name, f in impls.items():
        assert f(a, b) == exp, (name, a, b)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3269)
cnt = 0
for _ in range(400):
    a = [rng.randint(0, 1) for _ in range(rng.randint(0, 5))]
    b = [rng.randint(0, 1) for _ in range(rng.randint(1, 5))]
    exp = brute(a, b)
    for name, f in impls.items():
        assert f(a, b) == exp, (name, a, b)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cặp mảng.')

Khớp vét cạn trên 400 cặp mảng.


### 12.3. Ca lớn `1000 × 1000`

In [11]:
import time
big = {
    'ngẫu nhiên': ([rng.randint(0, 1) for _ in range(1000)], [rng.randint(0, 1) for _ in range(1000)]),
    'toàn 0': ([0] * 1000, [0] * 1000),
    'xen kẽ': ([i % 2 for i in range(1000)], [1 - i % 2 for i in range(1000)]),
}
for name, (a, b) in big.items():
    t = time.time(); x = Solution().minLargest(a, b); t1 = time.time() - t
    t = time.time(); y = table_dp(a, b); t2 = time.time() - t
    assert x == y
    print(f"{name:10s}: {x:5d} | nộp {t1:.2f}s | bảng {t2:.2f}s")
assert Solution().minLargest([0] * 1000, [0] * 1000) == 4000
print('toàn 0: cần 2000 số chẵn phân biệt -> 4000 — đúng.')

ngẫu nhiên:  2153 | nộp 0.32s | bảng 0.37s


toàn 0    :  4000 | nộp 0.20s | bảng 0.28s


xen kẽ    :  2000 | nộp 0.22s | bảng 0.27s
toàn 0: cần 2000 số chẵn phân biệt -> 4000 — đúng.


Cận dưới hiển nhiên: nếu tổng cộng cần `E` số chẵn và `O` số lẻ thì đáp án `≥ max(2E, 2O − 1)`; kiểm tra trên ca ngẫu nhiên.

In [12]:
for _ in range(200):
    a = [rng.randint(0, 1) for _ in range(rng.randint(0, 30))]
    b = [rng.randint(0, 1) for _ in range(rng.randint(1, 30))]
    E = (a + b).count(0); O = (a + b).count(1)
    assert Solution().minLargest(a, b) >= max(2 * E, 2 * O - 1)
print('đáp án >= max(2E, 2O - 1): đúng.')

đáp án >= max(2E, 2O - 1): đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Xử lý hai mảng độc lập (quên mỗi số chỉ dùng một lần)

In [13]:
def wrong_independent(a, b):
    def alone(arr):
        v = 0
        for x in arr:
            v = nxt(v, x)
        return v
    return max(alone(a), alone(b))                              # SAI

print('sai :', wrong_independent(*examples[1][:2]), '| đúng:', Solution().minLargest(*examples[1][:2]))

sai : 5 | đúng: 9


**Vì sao sai:** hai mảng dùng chung các số `2, 4, 5` — vi phạm "mỗi số dùng tối đa một lần".

### Lỗi 2 — Hàm `next` kiểm tra tính chẵn lẻ của `v` thay vì `v + 1`

In [14]:
def wrong_next(v, b):
    return v + 1 if v % 2 == b else v + 2                      # SAI: phải xét (v + 1) % 2

def wrong_next_dp(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = wrong_next(f[i - 1][0], a[i - 1])
    for j in range(1, n + 1):
        f[0][j] = wrong_next(f[0][j - 1], b[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = min(wrong_next(f[i - 1][j], a[i - 1]), wrong_next(f[i][j - 1], b[j - 1]))
    return f[m][n]

print('sai :', wrong_next_dp([], [0]), '| đúng:', Solution().minLargest([], [0]))

sai : 1 | đúng: 2


**Vì sao sai:** với `v = 0`, `b = 0` bản sai trả `1` — một số **lẻ** cho phần tử cần số chẵn. Điều kiện đúng là `(v + 1) % 2 == b`. Ô dưới cho thêm một phản ví dụ tìm tự động trên mảng ngẫu nhiên.

In [15]:
found = None
r_ = random.Random(1)
for _ in range(5000):
    a = [r_.randint(0, 1) for _ in range(r_.randint(0, 3))]; b = [r_.randint(0, 1) for _ in range(r_.randint(1, 3))]
    if wrong_next_dp(a, b) != brute(a, b):
        found = (a, b); break
print('phản ví dụ:', found, '-> sai:', wrong_next_dp(*found), '| đúng:', Solution().minLargest(*found))

phản ví dụ: ([0], [0, 1]) -> sai: 3 | đúng: 4


### Lỗi 3 — Tham lam khi trộn: luôn điền mảng có "số cần tiếp theo" nhỏ hơn

In [16]:
def wrong_greedy_merge(a, b):
    i = j = 0; v = 0
    while i < len(a) or j < len(b):
        ca = nxt(v, a[i]) if i < len(a) else float('inf')
        cb = nxt(v, b[j]) if j < len(b) else float('inf')
        if ca <= cb:                                             # SAI: nhìn một bước
            v = ca; i += 1
        else:
            v = cb; j += 1
    return v

found = None
r_ = random.Random(3)
for _ in range(5000):
    a = [r_.randint(0, 1) for _ in range(r_.randint(1, 4))]; b = [r_.randint(0, 1) for _ in range(r_.randint(1, 4))]
    if wrong_greedy_merge(a, b) != brute(a, b):
        found = (a, b); break
print('phản ví dụ:', found, '-> sai:', wrong_greedy_merge(*found), '| đúng:', Solution().minLargest(*found))

phản ví dụ: ([0, 0, 0, 0], [0, 1, 1]) -> sai: 13 | đúng: 10


**Vì sao sai:** khi hòa (cả hai mảng cần cùng tính chẵn lẻ), chọn mảng nào ảnh hưởng tới các bước sau — DP xét cả hai.

### Lỗi 4 — Dùng `max` thay vì `min` khi gộp hai nhánh

In [17]:
def wrong_max(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = nxt(f[i - 1][0], a[i - 1])
    for j in range(1, n + 1):
        f[0][j] = nxt(f[0][j - 1], b[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = max(nxt(f[i - 1][j], a[i - 1]), nxt(f[i][j - 1], b[j - 1]))   # SAI
    return f[m][n]

print('sai :', wrong_max(*examples[1][:2]), '| đúng:', Solution().minLargest(*examples[1][:2]))

sai : 13 | đúng: 9


**Vì sao sai:** ta được **chọn** phần tử nào điền cuối — lấy lựa chọn tốt hơn (nhỏ hơn).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hai dãy phải "chia nhau" một tài nguyên tăng dần (các số nguyên) ⇒ nghĩ tới **trộn hai dãy** và DP `(i, j)`.
- Giá trị trạng thái đơn điệu (nhỏ hơn luôn tốt hơn) ⇒ chỉ cần giữ giá trị tốt nhất mỗi trạng thái.
- Hàm "số kế tiếp thỏa điều kiện" giúp tham lam trong từng bước.

## 15. Biến thể của bài

1. Ba mảng ⇒ DP `(i, j, k)`.
2. Ràng buộc "chia hết cho 3 / dư 1 / dư 2" thay cho chẵn lẻ ⇒ đổi hàm `next`.
3. Không yêu cầu số phân biệt giữa hai mảng ⇒ `max` của hai mảng độc lập (lỗi 1 trở thành đúng).
4. Trả về cách gán cụ thể.

In [18]:
# Biến thể 2 (theo modulo 3) và 4 (dựng cách gán)
def next_mod(v, r, M=3):
    x = v + 1
    return x + (r - x) % M

def mod3_dp(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = next_mod(f[i - 1][0], a[i - 1])
    for j in range(1, n + 1):
        f[0][j] = next_mod(f[0][j - 1], b[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = min(next_mod(f[i - 1][j], a[i - 1]), next_mod(f[i][j - 1], b[j - 1]))
    return f[m][n]

def mod3_brute(a, b):
    m, n = len(a), len(b)
    M = 1
    while True:
        @lru_cache(maxsize=None)
        def go(v, i, j):
            if i == m and j == n:
                return True
            if v > M:
                return False
            return go(v + 1, i, j) or (i < m and v % 3 == a[i] and go(v + 1, i + 1, j)) or \
                (j < n and v % 3 == b[j] and go(v + 1, i, j + 1))
        if go(1, 0, 0):
            return M
        M += 1

def construct(a, b):
    m, n = len(a), len(b)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        f[i][0] = nxt(f[i - 1][0], a[i - 1])
    for j in range(1, n + 1):
        f[0][j] = nxt(f[0][j - 1], b[j - 1])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            f[i][j] = min(nxt(f[i - 1][j], a[i - 1]), nxt(f[i][j - 1], b[j - 1]))
    A, B = [0] * m, [0] * n
    i, j = m, n
    while i or j:
        if i and f[i][j] == nxt(f[i - 1][j], a[i - 1]):
            A[i - 1] = f[i][j]; i -= 1
        else:
            B[j - 1] = f[i][j]; j -= 1
    return A, B

rng = random.Random(15)
for _ in range(200):
    a = [rng.randint(0, 2) for _ in range(rng.randint(0, 4))]; b = [rng.randint(0, 2) for _ in range(rng.randint(1, 4))]
    assert mod3_dp(a, b) == mod3_brute(a, b)
    a2 = [x % 2 for x in a]; b2 = [x % 2 for x in b]
    A, B = construct(a2, b2)
    assert all(x % 2 == p for x, p in zip(A + B, a2 + b2)) and len(set(A + B)) == len(A + B)
    assert all(u < v for u, v in zip(A, A[1:])) and all(u < v for u, v in zip(B, B[1:]))
    assert max(A + B) == Solution().minLargest(a2, b2)
print('Biến thể mod 3 khớp vét cạn; dựng cách gán hợp lệ và đạt tối ưu. Ví dụ 2:', construct(*examples[1][:2]))

Biến thể mod 3 khớp vét cạn; dựng cách gán hợp lệ và đạt tối ưu. Ví dụ 2: ([6, 7, 8, 9], [1, 2, 4, 5])


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi cách dùng `1..M` | mũ/đa thức lớn | Oracle |
| 2 | Đệ quy có nhớ | `O(mn)` | Đệ quy sâu |
| 3 | Bảng DP | `O(mn)` | Dễ truy vết |
| 4 | DP lăn | `O(mn)`, `O(n)` bộ nhớ | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums1 = [1]`, `nums2 = [0]`.

**Bài 2 (tính tay):** `nums1 = [0, 0]`, `nums2 = [0]`.

**Bài 3 (điền TODO):** biến thể 1 — ba mảng.

**Bài 4 (tự viết lại):** biến thể 3 — số **không** cần phân biệt giữa hai mảng.

**Bài 5 (mini challenge):** đếm số cách gán đạt số lớn nhất tối ưu (vét cạn nhỏ).

In [19]:
def three_arrays_todo(a, b, c):
    # TODO
    raise NotImplementedError

def not_distinct_todo(a, b):
    # TODO
    raise NotImplementedError

def count_optimal_assignments_todo(a, b):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cần một số lẻ và một số chẵn. (2) `1` và `2`. (3) `2`.

**Bài 2.** (1) Ba số chẵn phân biệt. (2) `2, 4, 6`. (3) `6`.

**Bài 3.** (1) `f[i][j][k]`. (2) Ba nhánh. (3) `O(m n p)`.

**Bài 4.** (1) Hai mảng độc lập. (2) `max` của hai kết quả. (3) Chính là "lỗi 1".

**Bài 5.** (1) Vét cạn với `M` = đáp án: đếm số đường đi của `go`. (2) Mỗi đường đi là một cách gán. (3) Ví dụ nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 4
assert Solution().minLargest([1], [0]) == 2
assert Solution().minLargest([0, 0], [0]) == 6

def not_distinct(a, b):
    return wrong_independent(a, b)

print('Bài 1: 2 | Bài 2: 6 | Bài 4 (không phân biệt) ví dụ 2:', not_distinct(*examples[1][:2]))

Bài 1: 2 | Bài 2: 6 | Bài 4 (không phân biệt) ví dụ 2: 5


In [21]:
# Bài 3, 5
def three_arrays(a, b, c):
    m, n, p = len(a), len(b), len(c)
    INF = float('inf')
    f = [[[INF] * (p + 1) for _ in range(n + 1)] for _ in range(m + 1)]
    f[0][0][0] = 0
    for i in range(m + 1):
        for j in range(n + 1):
            for k in range(p + 1):
                v = f[i][j][k]
                if v == INF:
                    continue
                if i < m:
                    f[i + 1][j][k] = min(f[i + 1][j][k], nxt(v, a[i]))
                if j < n:
                    f[i][j + 1][k] = min(f[i][j + 1][k], nxt(v, b[j]))
                if k < p:
                    f[i][j][k + 1] = min(f[i][j][k + 1], nxt(v, c[k]))
    return f[m][n][p]

def three_brute(a, b, c):
    M = 1
    while True:
        @lru_cache(maxsize=None)
        def go(v, i, j, k):
            if i == len(a) and j == len(b) and k == len(c):
                return True
            if v > M:
                return False
            return go(v + 1, i, j, k) or (i < len(a) and v % 2 == a[i] and go(v + 1, i + 1, j, k)) or \
                (j < len(b) and v % 2 == b[j] and go(v + 1, i, j + 1, k)) or (k < len(c) and v % 2 == c[k] and go(v + 1, i, j, k + 1))
        if go(1, 0, 0, 0):
            return M
        M += 1

def count_optimal_assignments(a, b):
    M = Solution().minLargest(a, b)
    @lru_cache(maxsize=None)
    def ways(v, i, j):
        if v > M:
            return int(i == len(a) and j == len(b))
        total = ways(v + 1, i, j)
        if i < len(a) and v % 2 == a[i]:
            total += ways(v + 1, i + 1, j)
        if j < len(b) and v % 2 == b[j]:
            total += ways(v + 1, i, j + 1)
        return total
    return ways(1, 0, 0)

rng = random.Random(4)
for _ in range(150):
    a = [rng.randint(0, 1) for _ in range(rng.randint(0, 3))]
    b = [rng.randint(0, 1) for _ in range(rng.randint(1, 3))]
    c = [rng.randint(0, 1) for _ in range(rng.randint(0, 3))]
    assert three_arrays(a, b, c) == three_brute(a, b, c)
print('Bài 3 khớp vét cạn. Bài 5 ví dụ 1, 2:', count_optimal_assignments(*examples[0][:2]), count_optimal_assignments(*examples[1][:2]))

Bài 3 khớp vét cạn. Bài 5 ví dụ 1, 2: 1 12


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nx(v, p) = v + 1 nếu (v + 1) % 2 == p, ngược lại v + 2
row[j] = nx(row[j-1], b[j-1])                         # hàng 0
mỗi x trong a: new[0] = nx(row[0], x); new[j] = min(nx(row[j], x), nx(new[j-1], b[j-1]))
return row[n]
```

**Checklist:**

- [ ] Hai mảng dùng chung tập số (không xử lý độc lập)?
- [ ] `next` kiểm tra tính chẵn lẻ của `v + 1`?
- [ ] Gộp hai nhánh bằng `min`?
- [ ] `nums1` rỗng vẫn chạy (chỉ hàng 0)?
- [ ] Số chẵn **dương** nhỏ nhất là 2 (`next(0, 0) = 2`)?